# 02 Build dataset

**Owner:** _TBD_

Build the one dataset all models use: join OGT, replace each sequence by its ESM-2 embedding (480 dimensions), and fix the train/test split once.

**Reads:** `data/interim/meltome.parquet`, `data/raw/ogt.csv`, `data/interim/embeddings.npy`
**Writes:** `data/processed/dataset.parquet`, `data/processed/split.csv`

In [ ]:
import sys
from pathlib import Path

# Make `src` importable whether Jupyter starts in the repo root or in notebooks/.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import paths

In [ ]:
import hashlib
import subprocess

import numpy as np
import pandas as pd

from src.data import make_split

## 1. Merge and clean

In [ ]:
meltome = pd.read_parquet(paths.MELTOME)
ogt = pd.read_csv(paths.OGT)[["organism", "ogt"]]

df = meltome.merge(ogt, on="organism", how="left", validate="many_to_one")
assert df["ogt"].notna().all(), "Some organisms have no OGT yet: fill data/raw/ogt.csv (notebook 01)."
print(df.shape)

TODO: decide and document further cleaning, for example lysate vs. cell measurements, very short or very
long sequences, outliers in Tm.

## 2. ESM-2 embeddings

`scripts/embed_esm.py` computes one mean-pooled ESM-2 35M vector (480 dimensions) per unique sequence.
This takes about 3.6 hours on a laptop CPU and resumes after an interruption. The embeddings already exist:
copy `embeddings.npy`, `embeddings_index.parquet` and `embeddings.run.json` from the shared drive into
`data/interim/` instead of recomputing them.

In [ ]:
if not paths.EMBEDDINGS.exists():
    subprocess.run([sys.executable, str(ROOT / "scripts" / "embed_esm.py")], check=True)

vectors = np.load(paths.EMBEDDINGS)
index = pd.read_parquet(paths.EMBEDDINGS_INDEX)
print(vectors.shape)

## 3. Assemble the dataset

Columns: `id`, `organism`, `ogt`, `emb_0` … `emb_479`, `tm`.

In [ ]:
rows = df[["id", "sequence"]].merge(index, on="id", how="left", validate="one_to_one")
assert rows["embedding_row"].notna().all(), "Proteins without embedding: rerun scripts/embed_esm.py."
sequence_hash = rows["sequence"].map(lambda s: hashlib.sha256(s.encode()).hexdigest())
assert (sequence_hash == rows["sequence_sha256"]).all(), "Embeddings were computed from different sequences."

embeddings = pd.DataFrame(
    vectors[rows["embedding_row"].astype(int).to_numpy()],
    columns=[f"emb_{i}" for i in range(vectors.shape[1])],
)
dataset = pd.concat(
    [df[["id", "organism", "ogt"]].reset_index(drop=True), embeddings, df[["tm"]].reset_index(drop=True)],
    axis=1,
)
print(dataset.shape, "->", dataset.shape[1] - 3, "features (ogt + embeddings)")
dataset.head()

In [ ]:
paths.PROCESSED.mkdir(parents=True, exist_ok=True)
dataset.to_parquet(paths.DATASET, index=False)

## 4. Train/test split

Created once and shared by every model. Default: random 80/20, stratified by organism, seed 42.
The split is never overwritten automatically: delete `data/processed/split.csv` only if the team agrees.

In [ ]:
if paths.SPLIT.exists():
    print("Split exists, keeping it.")
else:
    make_split(dataset).to_csv(paths.SPLIT, index=False)

split = pd.read_csv(paths.SPLIT)
split["set"].value_counts()

## Notes and decisions

_Write down every choice made here (filters, parameters, problems) so the others can follow it._

-